In [1]:
from conquer3d.data_structure import TriangleMesh

import torch
import plotly.graph_objects as go
import conquer3d
import numpy as np

### 1. Create a Sphere

In [2]:
vertices, faces = conquer3d.creation.create_sphere(sectors=32, stacks=16, radius=1.0)
print(f"Vertices: {vertices.shape}, Faces: {faces.shape}")

Vertices: torch.Size([482, 3]), Faces: torch.Size([960, 3])


### 2. Create Triangle Mesh and Check Manifold
A closed sphere should be a perfect closed manifold (every edge has exactly 2 triangles).

In [3]:
mesh: TriangleMesh = conquer3d.data_structure.TriangleMesh(vertices.to('cuda'), faces.to('cuda'))
is_manifold_open = mesh.is_edge_manifold(allow_boundary_edge=True)
is_manifold_closed = mesh.is_edge_manifold(allow_boundary_edge=False)

print(f"Is Open Manifold (<=2 triangles per edge): {is_manifold_open}")
print(f"Is Closed Manifold (==2 triangles per edge): {is_manifold_closed}")

is_vertex_manifold = mesh.is_vertex_manifold()
print(f"Is Vertex Manifold: {is_vertex_manifold}")

# Compute areas and normals
areas = mesh.triangle_areas
normals = mesh.triangle_normals
surface_area = mesh.surface_area
print(f'Total Surface Area: {surface_area.item():.4f}')

# Compute face centers for normals
v_np = mesh.vertices.cpu().numpy()
f_np = mesh.triangles.cpu().numpy()
centers = np.mean(v_np[f_np], axis=1)
norms_np = normals.cpu().numpy()
areas_np = areas.cpu().numpy()

# Determine scaling factor for normals
scale = 0.1
tips = centers + norms_np * scale

# Build lines for normals. We interleave [center, tip, None] to draw multiple disconnected lines in one trace
lines_x = np.empty((len(centers) * 3,))
lines_y = np.empty((len(centers) * 3,))
lines_z = np.empty((len(centers) * 3,))
lines_x[0::3] = centers[:, 0]
lines_x[1::3] = tips[:, 0]
lines_x[2::3] = None
lines_y[0::3] = centers[:, 1]
lines_y[1::3] = tips[:, 1]
lines_y[2::3] = None
lines_z[0::3] = centers[:, 2]
lines_z[1::3] = tips[:, 2]
lines_z[2::3] = None

fig = go.Figure(data=[
    go.Mesh3d(
        x=v_np[:, 0], y=v_np[:, 1], z=v_np[:, 2],
        i=f_np[:, 0], j=f_np[:, 1], k=f_np[:, 2],
        intensity=areas_np, intensitymode='cell', colorscale='Viridis',
        opacity=0.8, showscale=True, name='Mesh'
    ),
    go.Scatter3d(
        x=lines_x, y=lines_y, z=lines_z,
        mode='lines', line=dict(color='black', width=2), showlegend=False
    ),
    go.Cone(
        x=tips[:, 0], y=tips[:, 1], z=tips[:, 2],
        u=norms_np[:, 0], v=norms_np[:, 1], w=norms_np[:, 2],
        sizemode="absolute", sizeref=scale*0.3, showscale=False, colorscale='Greys', anchor='tip'
    )
])

fig.update_layout(scene=dict(aspectmode='data'), title="Mesh with Areas and Normals")
fig.show()

Is Open Manifold (<=2 triangles per edge): True
Is Closed Manifold (==2 triangles per edge): True
Is Vertex Manifold: True
Total Surface Area: 12.4657


### 3. Ray Intersection
Cast 5 random rays from a point inside and a point outside the sphere, and highlight intersected triangles.

In [4]:
# Point inside (e.g., origin) and outside (e.g., [1, 1, 1])
inside_pt = torch.tensor([[0.0, 0.0, 0.0]], device='cuda', dtype=torch.float32).repeat(5, 1)
outside_pt = torch.tensor([[0.8, 0.8, 0.8]], device='cuda', dtype=torch.float32).repeat(20, 1)

ray_origins = torch.cat([inside_pt, outside_pt], dim=0)

# Random directions
torch.manual_seed(32)
ray_dirs = torch.randn(ray_origins.size(0), 3, device='cuda', dtype=torch.float32)
ray_dirs = torch.nn.functional.normalize(ray_dirs, dim=1)

# Perform Ray-Triangle Intersection
ray_ids, tri_ids, int_pts, distances = mesh.get_ray_intersection(ray_origins, ray_dirs, return_distance=True)

# Fetch vertices and faces for visualization
v_np = mesh.vertices.cpu().numpy()
f_np = mesh.triangles.cpu().numpy()

# Base mesh
fig = go.Figure(data=[
    go.Mesh3d(
        x=v_np[:, 0], y=v_np[:, 1], z=v_np[:, 2],
        i=f_np[:, 0], j=f_np[:, 1], k=f_np[:, 2],
        color='lightgray', opacity=1, showscale=False
    )
])

# Inside point rays (Indices 0 to 4) -> Blue
inside_mask = ray_ids < 5
inside_tris = tri_ids[inside_mask].cpu().numpy()
if len(inside_tris) > 0:
    for tri_idx in inside_tris:
        face = f_np[tri_idx]
        pts = v_np[face]
        fig.add_trace(go.Mesh3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            i=[0], j=[1], k=[2],
            color='blue', opacity=1.0
        ))

# Outside point rays (Indices 5 to 9) -> Red
outside_mask = ray_ids >= 5
outside_tris = tri_ids[outside_mask].cpu().numpy()
if len(outside_tris) > 0:
    for tri_idx in outside_tris:
        face = f_np[tri_idx]
        pts = v_np[face]
        fig.add_trace(go.Mesh3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            i=[0], j=[1], k=[2],
            color='red', opacity=1.0
        ))

# Draw the rays as lines
for i in range(ray_origins.size(0)):
    start = ray_origins[i].cpu().numpy()
    end = start + ray_dirs[i].cpu().numpy() * 1.5  # length 1.5
    color = 'blue' if i < 5 else 'red'
    fig.add_trace(go.Scatter3d(
        x=[start[0], end[0]], y=[start[1], end[1]], z=[start[2], end[2]],
        mode='lines', line=dict(color=color, width=3)
    ))

# Draw the intersection points with distances
int_pts_np = int_pts.cpu().numpy()
dist_np = distances.cpu().numpy()
fig.add_trace(go.Scatter3d(
    x=int_pts_np[:, 0], y=int_pts_np[:, 1], z=int_pts_np[:, 2],
    mode='markers+text',
    marker=dict(color='yellow', size=8, symbol='diamond'),
    text=[f"{d:.2f}" for d in dist_np],
    textposition="top center",
    textfont=dict(color='black', size=12),
    name='Intersections'
))

# Draw the points
fig.add_trace(go.Scatter3d(
    x=[0.0, 0.8], y=[0.0, 0.8], z=[0.0, 0.8],
    mode='markers', marker=dict(color=['blue', 'red'], size=6)
))

fig.update_layout(scene=dict(aspectmode='data'), title="Ray-Triangle Intersection")
fig.show()
